# Exploring `agent.py` — The Stateful Wrapper

This notebook is a hands-on walkthrough of `liteagent.Agent` — the stateful class that wraps
the raw loop. If the loop notebook explored the **engine**, this one explores the **car**.

**Why two layers?**

The raw loop (`agent_loop`, `agent_loop_continue`) is stateless — you pass in context, it returns
an EventStream, you iterate events yourself. It's the engine.

The `Agent` class wraps the loop and manages:
- **Message history** — accumulates across turns, no manual context threading
- **Event subscription** — `subscribe(callback)` instead of manual `async for`
- **Steering + follow-up queues** — `steer()` and `follow_up()` with dequeue modes
- **Cancellation** — `abort()` with partial message preservation
- **State tracking** — `is_streaming`, `stream_message`, `pending_tool_calls`, `error`

This is the same two-layer design as pi-mono (`agent-loop.ts` + `agent.ts`).
Most consumers will use `Agent`. Advanced consumers may use the raw loop directly.

**What we'll cover:**
1. Setup + imports
2. Simplest prompt — string in, messages out
3. `subscribe()` — the primary consumer API
4. `prompt()` overloads — string, dict, list, images
5. State access — what you can inspect during and after a run
6. Multi-turn — why Agent is stateful
7. Steering — `steer()` mid-run
8. Follow-up — `follow_up()` after idle
9. Queue modes — one-at-a-time vs all
10. `continue_run()` — resume from context
11. `abort()` and partial preservation
12. `wait_for_idle()`
13. `reset()` vs `clear_messages()`
14. Configuration setters — mid-run changes
15. Error handling
16. `_default_convert_to_llm` — what it does
17. Testing across models
18. Real-world patterns
19. Summary

---

## 1. Setup

The Agent needs a model string (litellm format) and optionally tools, system prompt,
and a `convert_to_llm` function. Let's import everything and define our helpers.

In [1]:
from liteagent import Agent, Tool, ToolResult
from liteagent.convert import make_default_convert

# Default model for all examples
MODEL = "anthropic/claude-sonnet-4-6"


# The Agent has a built-in default converter (make_default_convert) that:
# - Strips liteagent metadata (usage, timestamp, stop_reason, etc.)
# - Passes everything else through (thinking_blocks, reasoning_content,
#   provider_specific_fields — new litellm fields survive automatically)
# - For OpenAI models: hoists images from tool results into user messages
#   (OpenAI ignores image blocks in tool result content)
#
# Most examples below use the default — no convert_to_llm needed.
# We'll explore the converter in detail in section 16.


# Simple echo tool — reused across examples
async def echo_execute(tool_call_id, params, signal=None, on_update=None):
    return ToolResult(content=[{"type": "text", "text": params["message"]}])


echo_tool = Tool(
    name="echo",
    description="Echo back a message exactly",
    parameters={
        "type": "object",
        "properties": {
            "message": {"type": "string", "description": "The message to echo"}
        },
        "required": ["message"],
    },
    execute=echo_execute,
)

print("Setup complete.")

Setup complete.


## 2. Simplest prompt — string in, messages out

The absolute minimum: create an Agent, call `prompt("...")`, check `agent.messages`.

Unlike the raw loop (where you build `AgentContext` + `AgentConfig`, call `agent_loop`,
and iterate the EventStream yourself), the Agent does all of that internally.
`prompt()` blocks until the loop completes.

In [2]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise. One sentence max.",
)

await agent.prompt("What is 2 + 2?")

In [3]:
agent.messages

[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772973195640},
 {'role': 'assistant',
  'content': '4',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 25,
   'completion_tokens': 5,
   'total_tokens': 30,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973196928}]

In [4]:
agent.state.messages

[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772973195640},
 {'role': 'assistant',
  'content': '4',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 25,
   'completion_tokens': 5,
   'total_tokens': 30,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973196928}]

In [5]:
agent.state

AgentState(system_prompt='Be concise. One sentence max.', model='anthropic/claude-sonnet-4-6', thinking_level='off', tools=[], messages=[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772973195640}, {'role': 'assistant', 'content': '4', 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 25, 'completion_tokens': 5, 'total_tokens': 30, 'cache_read_tokens': 0, 'cache_creation_tokens': 0}, 'stop_reason': 'stop', 'timestamp': 1772973196928}], is_streaming=False, stream_message=None, pending_tool_calls=set(), error=None)

In [6]:
agent.state.messages

[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772973195640},
 {'role': 'assistant',
  'content': '4',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 25,
   'completion_tokens': 5,
   'total_tokens': 30,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973196928}]

In [8]:
# The default converter strips liteagent metadata, keeps LLM-compatible fields
convert = make_default_convert(MODEL)
convert(agent.state.messages)

[{'role': 'user', 'content': 'What is 2 + 2?'},
 {'role': 'assistant',
  'content': '4',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None}]

Two messages: the user prompt we sent, and the assistant's reply.
The Agent appended both to `agent.messages` automatically —
this is the key difference from the raw loop where you manage context yourself.

Let's look at the raw message objects:

In [9]:
# User message — our input, wrapped in a dict by prompt()
agent.messages[0]

{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772973195640}

In [10]:
# Assistant message — enriched with usage, stop_reason, timestamp
agent.messages[1]

{'role': 'assistant',
 'content': '4',
 'tool_calls': None,
 'thinking_blocks': None,
 'reasoning_content': None,
 'provider_specific_fields': None,
 'usage': {'prompt_tokens': 25,
  'completion_tokens': 5,
  'total_tokens': 30,
  'cache_read_tokens': 0,
  'cache_creation_tokens': 0},
 'stop_reason': 'stop',
 'timestamp': 1772973196928}

In [11]:
# This is why we usd @property --> Read-only access. With @property, ust prevents replacing the state object itself.
agent.state = "BREAK THE STATE"

AttributeError: property 'state' of 'Agent' object has no setter

The assistant message has all the extras the loop adds:
- `usage` — token counts from litellm
- `stop_reason` — "stop" (normal), "tool_calls", "error", "aborted"
- `timestamp` — Unix ms
- `thinking_blocks` / `reasoning_content` — None unless thinking is enabled
- `provider_specific_fields` — opaque bag from litellm

These extras are why `convert_to_llm` exists — they must be stripped before
sending messages back to the LLM.

## 3. `subscribe()` — the primary consumer API

In the loop notebook, we used `async for event in stream` to consume events.
The Agent doesn't expose the stream. Instead, you subscribe a callback:

```python
unsub = agent.subscribe(my_callback)  # returns unsubscribe function
```

The callback fires synchronously during `await agent.prompt()` — same thread,
no concurrency issues. This is how pi's agent works too.

**Why callbacks instead of async iteration?** The Agent is the sole reader of
the loop's EventStream (internal detail). External consumers get events via
subscribe — this lets multiple consumers see the same events (unlike a queue
where each item is consumed once).

In [12]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
)

# Collect all events
events = []
unsub = agent.subscribe(lambda e: events.append(e))

await agent.prompt("Say hello!")

In [13]:
for e in events:
    print(e)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'Say hello!', 'timestamp': 1772973294532}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'Say hello!', 'timestamp': 1772973294532}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'Hello! ', 'tool_calls': None}, 'delta': {'content': 'Hello! '}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'Hello! 👋 How are', 'tool_calls': None}, 'delta': {'content': '👋 How are'}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'Hello! 👋 How are you doing?', 'tool_calls': None}, 'delta': {'content': ' you doing?'}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'Hello! 👋 How are you doing? Is', 'tool_call

Same events sequence as the raw loop:

Now let's test unsubscribe:

In [14]:
count_before = len(events)
count_before

14

In [15]:
agent._subscribers

[<function __main__.<lambda>(e)>]

In [16]:
unsub()  # stop receiving events

In [17]:
agent._subscribers

[]

In [18]:
await agent.prompt("Say goodbye.")

count_after = len(events)
print(f"Events before unsub: {count_before}")
print(f"Events after second prompt: {count_after}")
print(f"Unsubscribe worked: {count_before == count_after}")

Events before unsub: 14
Events after second prompt: 14
Unsubscribe worked: True


In [19]:
agent.state.messages

[{'role': 'user', 'content': 'Say hello!', 'timestamp': 1772973294532},
 {'role': 'assistant',
  'content': 'Hello! 👋 How are you doing? Is there something I can help you with today?',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 15,
   'completion_tokens': 24,
   'total_tokens': 39,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973295408},
 {'role': 'user', 'content': 'Say goodbye.', 'timestamp': 1772973314512},
 {'role': 'assistant',
  'content': 'Goodbye! 👋 Take care, and feel free to come back anytime if you need anything! 😊',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 45,
   'completion_tokens': 29,
   'total_tokens': 74,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 17729733157

## 4. `prompt()` overloads

Like pi's `agent.prompt()`, ours accepts four input shapes:

| Input | What happens |
|-------|-------------|
| `prompt("string")` | Wrapped in `{"role": "user", "content": "string", "timestamp": ...}` |
| `prompt({"role": "user", ...})` | Used as-is |
| `prompt([msg1, msg2])` | Multiple messages injected |
| `prompt("text", images=[...])` | Multimodal: text + images in content array |

Let's see what each produces.

In [20]:
# Overload 1: string
agent = Agent(model=MODEL)
await agent.prompt("Hello from a string")
agent.messages[0]

{'role': 'user', 'content': 'Hello from a string', 'timestamp': 1772973328201}

In [21]:
# Overload 2: dict (used as-is)
agent = Agent(model=MODEL)
await agent.prompt(
    {"role": "user", "content": "Hello from a dict", "custom_field": "preserved"}
)
agent.messages[0]

{'role': 'user', 'content': 'Hello from a dict', 'custom_field': 'preserved'}

In [22]:
convert = make_default_convert(MODEL)
convert(agent.messages)

[{'role': 'user', 'content': 'Hello from a dict', 'custom_field': 'preserved'},
 {'role': 'assistant',
  'content': 'Hello! It looks like your message came through as plain text — "Hello from a dict" 👋\n\nCould you clarify what you mean? Are you:\n\n- **Referring to a Python dictionary?**\n- **Sending a greeting from a specific context or program?**\n- **Something else entirely?**\n\nLet me know and I\'ll be happy to help! 😊',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None}]

In [23]:
# Overload 3: list of messages
agent = Agent(model=MODEL, system_prompt="Be concise.")
await agent.prompt(
    [
        {"role": "user", "content": "My name is Alice."},
        {"role": "user", "content": "What is my name?"},
    ]
)

In [24]:
agent.messages

[{'role': 'user', 'content': 'My name is Alice.'},
 {'role': 'user', 'content': 'What is my name?'},
 {'role': 'assistant',
  'content': 'Your name is **Alice**! You just told me. 😊',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 22,
   'completion_tokens': 18,
   'total_tokens': 40,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973348804}]

In [25]:
# Overload 4: string + images (multimodal)
# Send a real image and ask the LLM about it

image_block = {
    "type": "image_url",
    "image_url": {
        "url": "https://dev-dashhudson-static.s3.amazonaws.com/research/media_asset_ai_generation/experiments/urbn_flat_lay/92207174_707_b3.jpg"
    },
}

agent = Agent(
    model=MODEL,
    system_prompt="Be concise. One sentence max.",
)
await agent.prompt("What is in this image?", images=[image_block])

print(f"Response: {agent.messages[-1].get('content')}")

Response: A yellow quilted tote bag with a red cherry print pattern and padded shoulder straps.


In [26]:
agent.messages

[{'role': 'user',
  'content': [{'type': 'text', 'text': 'What is in this image?'},
   {'type': 'image_url',
    'image_url': {'url': 'https://dev-dashhudson-static.s3.amazonaws.com/research/media_asset_ai_generation/experiments/urbn_flat_lay/92207174_707_b3.jpg'}}],
  'timestamp': 1772973355534},
 {'role': 'assistant',
  'content': 'A yellow quilted tote bag with a red cherry print pattern and padded shoulder straps.',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 1561,
   'completion_tokens': 23,
   'total_tokens': 1584,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973359005}]

## 5. State access

The Agent tracks state in an `AgentState` dataclass. You can inspect it at any time:

```python
agent.state.is_streaming       # True while loop is running
agent.state.stream_message     # current partial message being streamed (or None)
agent.state.pending_tool_calls # set of tool call IDs currently executing
agent.state.error              # last error message (or None)
agent.state.model              # current model string
agent.state.system_prompt      # current system prompt
agent.state.tools              # current tool list
agent.state.thinking_level     # "off", "minimal", "low", "medium", "high", "xhigh"
agent.messages                 # shorthand for agent.state.messages
```

Let's watch state change *during* a run using a subscriber:

In [27]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
    tools=[echo_tool],
)

# Track state transitions
state_log = []

print(
    f"{'Event':<26} {'Streaming':<10} {'StreamMsg':<10} {'PendTools':<10} {'MsgCount':<10}"
)


def track_state(event):
    t = event["type"]
    entry = {
        "event": t,
        "is_streaming": agent.state.is_streaming,
        "stream_msg": agent.state.stream_message is not None,
        "pending_tools": len(agent.state.pending_tool_calls),
        "msg_count": len(agent.messages),
    }
    state_log.append(entry)
    print("-" * 66)
    print(
        f"{entry['event']:<26} {str(entry['is_streaming']):<10} {str(entry['stream_msg']):<10} {entry['pending_tools']:<10} {entry['msg_count']:<10}"
    )


agent.subscribe(track_state)
await agent.prompt("Echo 'hello world'")

Event                      Streaming  StreamMsg  PendTools  MsgCount  
------------------------------------------------------------------
agent_start                True       False      0          0         
------------------------------------------------------------------
turn_start                 True       False      0          0         
------------------------------------------------------------------
message_start              True       False      0          0         
------------------------------------------------------------------
message_end                True       False      0          1         
------------------------------------------------------------------
message_start              True       True       0          1         
------------------------------------------------------------------
message_update             True       True       0          1         
------------------------------------------------------------------
message_update             True   

In [28]:
import pandas as pd

pd.DataFrame(state_log)

,event,is_streaming,stream_msg,pending_tools,msg_count
0,agent_start,True,False,0,0
1,turn_start,True,False,0,0
2,message_start,True,False,0,0
3,message_end,True,False,0,1
4,message_start,True,True,0,1
5,message_update,True,True,0,1
6,message_update,True,True,0,1
7,message_update,True,True,0,1
8,message_update,True,True,0,1
9,message_update,True,True,0,1


Notice how:
- `is_streaming` is True throughout the run
- `stream_message` appears on `message_start` (assistant only), disappears on `message_end`
- `pending_tool_calls` increments on `tool_execution_start`, decrements on `tool_execution_end`
- `msg_count` grows on each `message_end` — messages are appended incrementally, not batched


The tool call produces this pattern:
```
Turn 1: assistant calls echo → tool executes → tool result
Turn 2: assistant sees result → responds with text
```

Messages: user → assistant (tool_call) → tool (result) → assistant (text response)

## 6. Multi-turn — why Agent is stateful

This is the Agent's main value: messages persist across `prompt()` calls.
With the raw loop, you'd need to manually thread context between calls.
The Agent does it automatically.

In [29]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise. Remember everything the user says.",
)

# Turn 1: tell the agent something
await agent.prompt("My favorite color is blue.")
agent.messages

[{'role': 'user',
  'content': 'My favorite color is blue.',
  'timestamp': 1772973401784},
 {'role': 'assistant',
  'content': "Got it — your favorite color is blue! I'll remember that. Is there something I can help you with?",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 24,
   'completion_tokens': 26,
   'total_tokens': 50,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973403290}]

In [30]:
# Turn 2: ask about it — the agent should remember
await agent.prompt("What is my favorite color?")
agent.messages

[{'role': 'user',
  'content': 'My favorite color is blue.',
  'timestamp': 1772973401784},
 {'role': 'assistant',
  'content': "Got it — your favorite color is blue! I'll remember that. Is there something I can help you with?",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 24,
   'completion_tokens': 26,
   'total_tokens': 50,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973403290},
 {'role': 'user',
  'content': 'What is my favorite color?',
  'timestamp': 1772973404680},
 {'role': 'assistant',
  'content': 'Your favorite color is blue!',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 59,
   'completion_tokens': 9,
   'total_tokens': 68,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 17

## 7. Steering — `steer()` mid-run

`steer()` queues a message that gets injected **during** a run:
- After each tool execution, the loop checks the steering queue
- If there's a message, remaining tools are **skipped** and the steering message
  is injected before the next LLM call
- This is "stop what you're doing, do this instead"

The loop also checks for steering at the start of each run (before the first LLM call).
So if you call `steer()` before `prompt()`, the steering message gets picked up immediately.

Let's demonstrate both: pre-queued steering, and mid-tool steering.

In [31]:
# Pre-queued steering: steer() before prompt()
agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
)

agent.steer("Actually, tell me a joke instead.")
await agent.prompt("What is the capital of France?")

agent.messages

[{'role': 'user',
  'content': 'What is the capital of France?',
  'timestamp': 1772973409234},
 {'role': 'user',
  'content': 'Actually, tell me a joke instead.',
  'timestamp': 1772973409234},
 {'role': 'assistant',
  'content': "Sure! Here's one:\n\nWhy don't scientists trust atoms?\n\n**Because they make up everything!** 😄",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 27,
   'completion_tokens': 29,
   'total_tokens': 56,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973410610}]

In [32]:
# Mid-tool steering: steer() during tool execution
# When tool_a executes, it queues a steering message.
# tool_b should be SKIPPED.

call_log = []
steering_agent = None  # forward reference


async def tool_a_exec(tool_call_id, params, signal=None, on_update=None):
    call_log.append("a")
    steering_agent.steer("Stop! Do something else.")  # interrupt!
    return ToolResult(content=[{"type": "text", "text": "tool_a done"}])


async def tool_b_exec(tool_call_id, params, signal=None, on_update=None):
    call_log.append("b")
    return ToolResult(content=[{"type": "text", "text": "tool_b done"}])


tool_a = Tool(
    name="tool_a",
    description="Tool A",
    parameters={"type": "object", "properties": {}},
    execute=tool_a_exec,
)
tool_b = Tool(
    name="tool_b",
    description="Tool B",
    parameters={"type": "object", "properties": {}},
    execute=tool_b_exec,
)

steering_agent = Agent(
    model=MODEL,
    system_prompt="When asked, call both tool_a and tool_b in a single response. Be concise.",
    tools=[tool_a, tool_b],
)

await steering_agent.prompt("Call both tool_a and tool_b now.")

steering_agent.messages

[{'role': 'user',
  'content': 'Call both tool_a and tool_b now.',
  'timestamp': 1772973413558},
 {'role': 'assistant',
  'content': 'Sure! Calling both tools simultaneously right away!',
  'tool_calls': [{'id': 'toolu_013sG8S32ceUQgyXCLXLAaLm',
    'type': 'function',
    'function': {'name': 'tool_a', 'arguments': '{}'},
    'provider_specific_fields': None},
   {'id': 'toolu_01Czzm2NF9JDFSw2FgVXsCwF',
    'type': 'function',
    'function': {'name': 'tool_b', 'arguments': '{}'},
    'provider_specific_fields': None}],
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 609,
   'completion_tokens': 66,
   'total_tokens': 675,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'tool_calls',
  'timestamp': 1772973414899},
 {'role': 'tool',
  'tool_call_id': 'toolu_013sG8S32ceUQgyXCLXLAaLm',
  'name': 'tool_a',
  'content': [{'type': 'text', 'text': 'tool_a done'}],
  'details': {},
  'is_erro

  1. Assistant calls both tool_a and tool_b
  2. tool_a executes (and queues steering inside its execute function)
  3. tool_b gets skipped — is_error: True, "Skipped due to queued user message."
  4. Steering message injected: "Stop! Do something else."
  5. Assistant responds to the steering instead of continuing

  The key proof: tool_b never ran ('b' not in call_log), but it still has a tool result in the conversation — that's the synthetic skip result from _skip_tool_call() in
  loop.py:94-129. The LLM needs every tool call to have a result, even skipped ones.

## 8. Follow-up — `follow_up()` after idle

`follow_up()` is the *outer loop* mechanism. Unlike steering (which interrupts),
follow-ups wait until the agent finishes everything (no more tool calls, no steering).
Then the follow-up message is injected and the agent continues.

- Steering = "stop what you're doing" (immediate)
- Follow-up = "when you're done, also do this" (deferred)

In [33]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise. One sentence.",
)

# Queue a follow-up BEFORE the first prompt
agent.follow_up("Now tell me a fun fact about cats.")

await agent.prompt("What is 2 + 2?")

agent.messages

[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772973526819},
 {'role': 'assistant',
  'content': '4',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 24,
   'completion_tokens': 5,
   'total_tokens': 29,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973527419},
 {'role': 'user',
  'content': 'Now tell me a fun fact about cats.',
  'timestamp': 1772973526819},
 {'role': 'assistant',
  'content': "Cats can't taste sweetness because they lack the taste receptors for it.",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 41,
   'completion_tokens': 20,
   'total_tokens': 61,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973528781}]

## 9. Queue modes

Both steering and follow-up have two modes:
- `"one-at-a-time"` (default) — dequeue one message per poll
- `"all"` — dequeue everything at once

This matters when multiple messages are queued. Let's see the difference.

In [34]:
# one-at-a-time (default): queue 3, dequeue returns 1
agent = Agent(model=MODEL)
agent.steer("msg1")
agent.steer("msg2")
agent.steer("msg3")

batch = agent._dequeue_steering()
print(
    f"one-at-a-time: got {len(batch)} message(s), {len(agent._steering_queue)} remaining"
)
print(f"  dequeued: '{batch[0]['content']}'")

one-at-a-time: got 1 message(s), 2 remaining
  dequeued: 'msg1'


In [35]:
# all mode: queue 3, dequeue returns all 3
agent = Agent(model=MODEL, steering_mode="all")
agent.steer("msg1")
agent.steer("msg2")
agent.steer("msg3")

batch = agent._dequeue_steering()
print(f"all mode: got {len(batch)} message(s), {len(agent._steering_queue)} remaining")
for m in batch:
    print(f"  '{m['content']}'")

all mode: got 3 message(s), 0 remaining
  'msg1'
  'msg2'
  'msg3'


## 10. `continue_run()` — resume from context

`continue_run()` is for when the conversation ended at a tool result or user message
and you want the LLM to continue from there — without sending a new prompt.

Three interesting cases when the last message is an assistant message:
1. Steering queue has messages → use those
2. Follow-up queue has messages → use those
3. Both empty → error (can't continue from assistant without new input)

**When would you actually use this?** In normal chat (`prompt()` → response → `prompt()` again),
you won't. `continue_run()` is for **recovery and resumption** — when something outside the
normal flow modifies the message history. Real-world examples from pi-mono's coding agent:

- **Context compaction** — when the conversation gets too long, old messages are summarized and
  replaced. After compaction the last message might be a tool result the LLM never responded to.
  `continue()` kicks the loop to pick up from there.
- **Error retry** — when an LLM call fails, a "please retry" message is appended and `continue()`
  re-runs the loop without needing a new user prompt.
- **Queued messages after idle** — if `follow_up()` or `steer()` is called after the agent
  finishes, `continue_run()` restarts the loop to process them.

In [36]:
# Case: continue from a tool result (manually built context)
agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
)

# Simulate: user asked about weather → assistant called tool → we have the result
agent._state.messages = [
    {"role": "user", "content": "What's the weather?"},
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "id": "c0",
                "type": "function",
                "function": {"name": "weather", "arguments": "{}"},
            }
        ],
        "stop_reason": "tool_calls",
    },
    {
        "role": "tool",
        "tool_call_id": "c0",
        "content": [{"type": "text", "text": "72°F and sunny in San Francisco"}],
        "is_error": False,
    },
]

await agent.continue_run()

agent.messages

[{'role': 'user', 'content': "What's the weather?"},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'c0',
    'type': 'function',
    'function': {'name': 'weather', 'arguments': '{}'}}],
  'stop_reason': 'tool_calls'},
 {'role': 'tool',
  'tool_call_id': 'c0',
  'content': [{'type': 'text', 'text': '72°F and sunny in San Francisco'}],
  'is_error': False},
 {'role': 'assistant',
  'content': "I'm sorry, but I don't have a weather tool available to check current conditions. Could you please:\n\n1. **Share your location** – so I can point you to the right resource.\n2. **Check a weather service** – such as:\n   - [weather.com](https://www.weather.com)\n   - [Google Weather](https://www.google.com/search?q=weather)\n   - A weather app on your phone\n\nWould you like help with anything else?",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 609,
   'completion_tokens': 112,
  

In [37]:
# Case: continue from assistant + steering queue
agent = Agent(model=MODEL)
agent._state.messages = [
    {"role": "user", "content": "Hi"},
    {"role": "assistant", "content": "Hello!", "stop_reason": "stop"},
]
agent.steer("Now tell me a joke.")

await agent.continue_run()

agent.messages

[{'role': 'user', 'content': 'Hi'},
 {'role': 'assistant', 'content': 'Hello!', 'stop_reason': 'stop'},
 {'role': 'user',
  'content': 'Now tell me a joke.',
  'timestamp': 1772973585526},
 {'role': 'assistant',
  'content': "Sure! Here's one:\n\nWhy don't scientists trust atoms?\n\nBecause they make up everything! 😄\n\nWant to hear another one?",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 22,
   'completion_tokens': 34,
   'total_tokens': 56,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973587023}]

In [38]:
# Case: continue from assistant + empty queues → error
agent = Agent(model=MODEL)
agent._state.messages = [
    {"role": "user", "content": "Hi"},
    {"role": "assistant", "content": "Hello!", "stop_reason": "stop"},
]

try:
    await agent.continue_run()
except ValueError as e:
    print(f"Got expected error: {e}")

Got expected error: Cannot continue from assistant message without queued messages. Use steer() or follow_up() first.


## 11. `abort()` and partial preservation

`abort()` sets a signal that stops the loop. But what happens to the partial
assistant message that was being streamed? The Agent handles this edge case
(same as pi's agent.ts lines 504-518):

1. If the partial has **real content** (non-empty text, reasoning, or named tool call) → **preserve it**
2. If it's just **empty scaffolding** (empty strings, unnamed tool calls) → **discard it**
3. If discarded after abort → raise "Request was aborted" → caught by error handler

In [39]:
# abort() after receiving some text — partial should be preserved
agent = Agent(
    model=MODEL,
    system_prompt="Write a very long essay about the history of computing. At least 5000 words.",
)

chunk_count = 0


def abort_after_chunks(event):
    print("signal is set: " + str(agent._signal.is_set()))
    if agent.state.stream_message:
        print(agent.state.stream_message["content"])
    global chunk_count
    if event["type"] == "message_update" and event.get("delta_type") == "text_delta":
        chunk_count += 1
        if chunk_count >= 5:
            agent.abort()


agent.subscribe(abort_after_chunks)
await agent.prompt("Go ahead.")

print(f"is_streaming: {agent.state.is_streaming}")
print(f"signal cleaned up: {agent._signal is None}")

signal is set: False
signal is set: False
signal is set: False
signal is set: False
signal is set: False
None
signal is set: False
# The History
signal is set: False
# The History of Computing: From Ancient Ab
signal is set: False
# The History of Computing: From Ancient Abacus to
signal is set: False
# The History of Computing: From Ancient Abacus to Artificial
signal is set: False
# The History of Computing: From Ancient Abacus to Artificial Intelligence

## A
signal is set: True
# The History of Computing: From Ancient Abacus to Artificial Intelligence

## A Comprehensive Survey
signal is set: True
# The History of Computing: From Ancient Abacus to Artificial Intelligence

## A Comprehensive Survey of
signal is set: True
# The History of Computing: From Ancient Abacus to Artificial Intelligence

## A Comprehensive Survey of Humanity
signal is set: True
# The History of Computing: From Ancient Abacus to Artificial Intelligence

## A Comprehensive Survey of Humanity's Most Transform
s

In [40]:
agent.messages

[{'role': 'user', 'content': 'Go ahead.', 'timestamp': 1772973598834},
 {'role': 'assistant',
  'content': "# The History of Computing: From Ancient Abacus to Artificial Intelligence\n\n## A Comprehensive Survey of Humanity's Most Transform",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 0,
   'completion_tokens': 28,
   'total_tokens': 28,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'aborted',
  'timestamp': 1772973600721}]

## 12. `wait_for_idle()`

  A coordination primitive. `prompt()` already awaits internally, so when it returns
  the agent is idle. `wait_for_idle()` is for when something *else* triggered the agent
  and you need to sync from a different place in your code:

  ```python
  # Some callback triggered a run
  agent.follow_up("do something")
  asyncio.create_task(agent.continue_run())

  # Later, elsewhere:
  await agent.wait_for_idle()   # block until that run finishes
  # now safe to inspect agent.messages

  If you're always doing await agent.prompt() sequentially, you'll never need this.

In [41]:
# When idle, returns immediately
agent = Agent(model=MODEL)
await agent.wait_for_idle()  # should not hang
print("wait_for_idle() returned immediately (agent is idle)")

# After a prompt, also returns immediately (prompt already blocks)
await agent.prompt("Hi")
await agent.wait_for_idle()
print("wait_for_idle() returned immediately (prompt already completed)")

wait_for_idle() returned immediately (agent is idle)
wait_for_idle() returned immediately (prompt already completed)


## 13. `reset()` vs `clear_messages()`

Two ways to clear state, with different scopes:

| Method | Clears messages | Clears queues | Clears error | Keeps config |
|--------|:-:|:-:|:-:|:-:|
| `reset()` | ✓ | ✓ | ✓ | ✓ |
| `clear_messages()` | ✓ | ✗ | ✗ | ✓ |

`reset()` is "start over". `clear_messages()` is "clear history but keep queued work".

In [42]:
agent = Agent(
    model=MODEL,
    system_prompt="you are a nice agent",
    tools=[echo_tool],
)
agent.append_message({"role": "user", "content": "old message"})
agent.steer("queued steering")
agent.follow_up("queued follow-up")
agent._state.error = "some error"

print("Before clear_messages():")
agent.messages

Before clear_messages():


[{'role': 'user', 'content': 'old message'}]

In [43]:
agent.clear_messages()

print("After clear_messages():")
agent.messages

After clear_messages():


[]

In [44]:
# Now reset — clears everything
agent.append_message({"role": "user", "content": "new message"})
agent.reset()

print("After reset():")
print(
    f"  messages: {len(agent.messages)}, queued: {agent.has_queued_messages()}, error: {agent.state.error}"
)
print(f"  model: {agent.state.model}  ← preserved")
print(f"  system_prompt: '{agent.state.system_prompt}'  ← preserved")
print(f"  tools: {len(agent.state.tools)}  ← preserved")

After reset():
  messages: 0, queued: False, error: None
  model: anthropic/claude-sonnet-4-6  ← preserved
  system_prompt: 'you are a nice agent'  ← preserved
  tools: 1  ← preserved


## 14. Configuration setters — mid-run changes

Pi allows calling `setModel()`, `setTools()`, etc. even while the agent is streaming.
The loop snapshots context at the start of each run, so mid-run changes only take
effect on the **next** run. We match this behavior — no streaming guard.

This enables patterns like:
- **Dynamic capabilities:** escalate tool permissions mid-conversation
- **Model switching:** use a cheap model for simple tasks, expensive for complex ones
- **Plan mode:** swap tool sets between planning and execution phases

In [45]:
agent = Agent(model=MODEL)


# Track which model gets called
def event_handler(e):
    if e["type"] == "message_end":
        print(f"Model Used: {agent.state.model}")


agent.subscribe(event_handler)

await agent.prompt("hey")

# Switch model mid-conversation
agent.set_model("gemini/gemini-3-flash-preview")

await agent.prompt("Bye")
agent.messages

Model Used: anthropic/claude-sonnet-4-6
Model Used: anthropic/claude-sonnet-4-6
Model Used: gemini/gemini-3-flash-preview
Model Used: gemini/gemini-3-flash-preview


[{'role': 'user', 'content': 'hey', 'timestamp': 1772973671100},
 {'role': 'assistant',
  'content': "Hey! How's it going? What's on your mind? 😊",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 8,
   'completion_tokens': 19,
   'total_tokens': 27,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772973672319},
 {'role': 'user', 'content': 'Bye', 'timestamp': 1772973672320},
 {'role': 'assistant',
  'content': 'Bye! Have a great rest of your day. Feel free to reach out if you need anything later! 👋',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': {'thought_signatures': ['EjQKMgG+Pvb7m/lEYBiQTYwu+gbwvlZJdvawahUWkMVeZexzMHKDYv3yqkH+vFPlUMTHye5y']},
  'usage': {'prompt_tokens': 21,
   'completion_tokens': 22,
   'total_tokens': 43,
   'cache_read_tokens': 0,
   'cache_creation_tokens'

## 15. Error handling

When the LLM call fails (network error, rate limit, etc.), the Agent:
1. Catches the exception
2. Creates a synthetic assistant message with `stop_reason="error"`
3. Appends it to messages
4. Sets `agent.state.error`
5. Emits `agent_end` event
6. Cleans up (is_streaming=False, etc.)

The Agent does **not** re-raise — it always completes cleanly. This lets consumers
check `agent.state.error` instead of wrapping every `prompt()` in try/except.

In [46]:
# Force an error by using a non-existent model
agent = Agent(model="fake-provider/nonexistent-model")
await agent.prompt("This will fail.")

agent.messages


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



[{'role': 'user', 'content': 'This will fail.', 'timestamp': 1772973711223},
 {'role': 'assistant',
  'content': None,
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'model': 'fake-provider/nonexistent-model',
  'usage': {'prompt_tokens': 0,
   'completion_tokens': 0,
   'total_tokens': 0,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'error',
  'error_message': "litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=fake-provider/nonexistent-model\n Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers",
  'timestamp': 1772973711228}]

## 16. `make_default_convert` — the default converter

If you don't provide `convert_to_llm`, the Agent builds one via
`make_default_convert(model)` from `liteagent/convert.py`.

This is the **sole provider-specific boundary** in the codebase. It uses a
**denylist** approach: strip known liteagent metadata fields, pass everything
else through. This means new fields litellm adds (like future provider-specific
attributes) survive automatically — no allowlist to update.

**What it strips:** `timestamp`, `usage`, `stop_reason`, `error_message`,
`details`, `is_error`

**What it preserves:** `thinking_blocks`, `reasoning_content`,
`provider_specific_fields`, multimodal content blocks, tool call metadata —
everything the LLM needs for multi-turn fidelity.

**OpenAI image hoisting:** For OpenAI models, tool results with `[text, image_url]`
content get split — text stays in the tool message, images are hoisted into a
synthetic user message. This is because OpenAI's Chat Completions API silently
ignores image blocks in tool result content. Anthropic and Gemini handle them
natively, so no hoisting needed.

See `learnings/LITELLM_API_LANDSCAPE.md` for the full investigation.

In [47]:
from liteagent.convert import make_default_convert

# Build the converter for Anthropic (no image hoisting needed)
convert_anthropic = make_default_convert("anthropic/claude-sonnet-4-6")

# Build one for OpenAI (will hoist tool-result images)
convert_openai = make_default_convert("gpt-5.2")

# Simulate a conversation with enriched messages
messages = [
    {"role": "user", "content": "Hi", "timestamp": 12345},
    {
        "role": "assistant",
        "content": "Hello!",
        "tool_calls": None,
        "thinking_blocks": [{"type": "thinking", "thinking": "greeting"}],
        "reasoning_content": "simple greeting",
        "provider_specific_fields": {"thought_signatures": ["sig123"]},
        "usage": {"prompt_tokens": 10, "completion_tokens": 5},
        "stop_reason": "stop",
        "timestamp": 12346,
    },
    {
        "role": "tool",
        "tool_call_id": "c0",
        "name": "chart",
        "content": [
            {"type": "text", "text": "Here is the chart."},
            {"type": "image_url", "image_url": {"url": "data:image/png;base64,abc"}},
        ],
        "is_error": False,
        "details": {"extra": "ui-only"},
        "timestamp": 12347,
    },
]

print("=== Anthropic converter (images stay in tool message) ===")
for m in convert_anthropic(messages):
    print(m)

print()
print("=== OpenAI converter (images hoisted to user message) ===")
for m in convert_openai(messages):
    print(m)

=== Anthropic converter (images stay in tool message) ===
{'role': 'user', 'content': 'Hi'}
{'role': 'assistant', 'content': 'Hello!', 'tool_calls': None, 'thinking_blocks': [{'type': 'thinking', 'thinking': 'greeting'}], 'reasoning_content': 'simple greeting', 'provider_specific_fields': {'thought_signatures': ['sig123']}}
{'role': 'tool', 'tool_call_id': 'c0', 'name': 'chart', 'content': [{'type': 'text', 'text': 'Here is the chart.'}, {'type': 'image_url', 'image_url': {'url': 'data:image/png;base64,abc'}}]}

=== OpenAI converter (images hoisted to user message) ===
{'role': 'user', 'content': 'Hi'}
{'role': 'assistant', 'content': 'Hello!', 'tool_calls': None, 'thinking_blocks': [{'type': 'thinking', 'thinking': 'greeting'}], 'reasoning_content': 'simple greeting', 'provider_specific_fields': {'thought_signatures': ['sig123']}}
{'role': 'tool', 'tool_call_id': 'c0', 'name': 'chart', 'content': 'Here is the chart.'}
{'role': 'user', 'content': [{'type': 'text', 'text': 'Image from t

The default converter handles the common cases transparently. You'd override it when:

1. **Custom message types** — your app stores `{"role": "notification", ...}` that need
   to be filtered or converted
2. **Custom compaction** — you want to summarize old messages before sending
3. **Provider-specific optimizations** — e.g., different image handling for a specific model

For most uses, the default just works — you never need to pass `convert_to_llm`.

## 17. Testing across models

The Agent is model-agnostic. Let's run the same prompt + tool call through some different models.

In [48]:
MODELS = [
      "anthropic/claude-sonnet-4-6",
      "anthropic/claude-opus-4-6",
      "gemini/gemini-3-pro-preview",
      "gemini/gemini-3-flash-preview",
      "gpt-5.2",
      "gpt-5.3-codex",
      "gpt-5.4"]

for model in MODELS:
    agent = Agent(
        model=model,
        system_prompt="Use the echo tool. Be concise.",
        tools=[echo_tool],
    )

    try:
        await agent.prompt("Echo 'test'")
        roles = [m.get("role") for m in agent.messages]
        has_tool = "tool" in roles
        assistants = [m for m in agent.messages if m.get("role") == "assistant"]
        last_content = (assistants[-1].get("content") or "")[:50] if assistants else "?"
        print(f"  ✓ {model:<42} tool_used={has_tool} | {last_content}")
    except Exception as e:
        print(f"  ✗ {model:<42} ERROR: {e}")

  ✓ anthropic/claude-sonnet-4-6                tool_used=True | The echoed message is: **test**
  ✓ anthropic/claude-opus-4-6                  tool_used=True | Done! The message "test" was echoed back.
  ✓ gemini/gemini-3-pro-preview                tool_used=True | test
  ✓ gemini/gemini-3-flash-preview              tool_used=True | test
  ✓ gpt-5.2                                    tool_used=True | 'test'


/Users/christopher/personal_projects/liteagent/.venv/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ResponseAPIUsage` - serialized value may not be as expected [field_name='usage', input_value={'completion_tokens': 17,..., 'image_tokens': None}}, input_type=dict])
  return self.__pydantic_serializer__.to_python(
/Users/christopher/personal_projects/liteagent/.venv/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ResponseAPIUsage` - serialized value may not be as expected [field_name='usage', input_value={'completion_tokens': 5, ..., 'image_tokens': None}}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


  ✓ gpt-5.3-codex                              tool_used=True | test
  ✓ gpt-5.4                                    tool_used=True | test


## 18. Real-world patterns

The Agent is framework-agnostic. Here's how you'd wire it into different consumers.

In [49]:
# Pattern 1: CLI — print text deltas as they arrive

agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
)


def cli_handler(event):
    if event["type"] == "message_update" and event.get("delta_type") == "text_delta":
        text = event["delta"].get("content", "")
        print(text, end="", flush=True)
    elif event["type"] == "agent_end":
        print()  # newline at the end


agent.subscribe(cli_handler)
print("Agent: ", end="")
await agent.prompt("What is the meaning of life, in one sentence?")

Agent: The meaning of life is whatever purpose, connection, and fulfillment you consciously choose to create and pursue.
